In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib

In [ ]:
CLEAN_TMDB_FILE_PATH = "../datasets/clean/tmdb-movies/TMDB_movie_dataset_v11.csv"
CLEAN_MOVIELENS_RATINGS_PATH = "../datasets/clean/ml-32m/ratings.csv"

ML_API_TF_IDF_MATRIX_PATH = "../../ml-api/model/tf_idf_matrix.pkl"

In [ ]:
tmdb = pd.read_csv(CLEAN_TMDB_FILE_PATH)
ratings = pd.read_csv(CLEAN_MOVIELENS_RATINGS_PATH)

In [ ]:
ratings_glob_mean = ratings["rating"].mean()
tmdb_id_to_index = pd.Series(tmdb.index, index=tmdb["id"]).to_dict()
ratings["rating"] = ratings["rating"] - ratings_glob_mean

In [ ]:
plt.hist(ratings["rating"], bins=50)
plt.title("Distribution ratings")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print("min:", ratings["rating"].min())
print("max:", ratings["rating"].max())
print("mean:", ratings["rating"].mean())
print("std:", ratings["rating"].std())
print(np.percentile(ratings["rating"], [0, 25, 50, 75, 100]))

In [ ]:
def find_in_dataset_by_substring(movie_names):
    found = []
    for movie_name in movie_names:
        results = tmdb[tmdb["title"].str.contains(movie_name, case=False, na=False)]
        haa = results.sort_values(by="popularity", ascending=False)[["id", "title"]]
        found.append(haa.values.tolist())
    return found

def find_in_dataset_by_id(movie_id):
    return tmdb[tmdb["id"] == movie_id]["title"].values[0]

def print_ratings_dict(ratings_dict: dict[int, int]):
    for id, rating in ratings_dict.items():
        print(f"id: {id}, name: {find_in_dataset_by_id(id)}, rating: {rating}")

In [ ]:
find_in_dataset_by_substring(["justice", "batman", "superman", "flash", "lantern", "steel", "watchmen", "joker"])

In [ ]:
find_in_dataset_by_id(791373)

In [ ]:
marvel_fan_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  26881: 4,
  299536: 5,
  9320: 4.5
}

print_ratings_dict(marvel_fan_rd)
print()
print()
print()

marvel_fan_dc_hater_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  26881: 4,
  299536: 5,
  9320: 4.5,
  141052: 1,
  209112: 0,
  414906: 2.5,
  272: 1.4,
  1924: 1,
  298618: 0.7,
  44912: 0,
  49521: 1.8,
  13183: 0.3,
  475557: 3
}

print_ratings_dict(marvel_fan_dc_hater_rd)

Collaborative filtering


In [ ]:
ratings[ratings["tmdbId"].isna()]

In [ ]:
ratings[ratings["userId"].isna()]

In [ ]:
movie_stats = ratings.groupby("tmdbId").agg(
    avg=("rating", "mean"),
    count=("rating", "count")
)

good_movies = movie_stats[
    (movie_stats["count"] >= 0)
].index

ratings_filtered = ratings[ratings["tmdbId"].isin(good_movies)]

In [ ]:
len(np.sort(ratings["tmdbId"].unique()))

In [ ]:
ratings_test = ratings_filtered


user_ids = np.sort(ratings_test["userId"].unique())
movie_ids = np.sort(ratings_test["tmdbId"].unique())

user_map = {u: i for i, u in enumerate(user_ids)}
movie_map = {m: i for i, m in enumerate(movie_ids)}

n_users = len(user_ids)
n_items = len(movie_ids)

rows = ratings_test["userId"].map(user_map)
cols = ratings_test["tmdbId"].map(movie_map)
data = ratings_test["rating"]

R = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

svd = TruncatedSVD(n_components=35)
U = svd.fit_transform(R)
V = svd.components_

In [ ]:
U

In [ ]:
V

In [ ]:
def build_user_profile_collab(ratings_dict):
    _indices = []
    _ratings = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in movie_map:
            _indices.append(movie_map[tmdb_id])
            _ratings.append(rating - ratings_glob_mean)
        
    V_sub = V[:, _indices]
    r = np.array(_ratings)
    user_vector = r @ V_sub.T
    return user_vector

In [ ]:
def recommend_content(user_vector, top_k=20):
    scores = user_vector @ V

    top_idx = scores.argsort()[::-1][:top_k]
    top_movies = [(movie_ids[i], scores[i]) for i in top_idx]

    results = []

    tmdb_indexed = tmdb.set_index("id")

    for movie_id, score in top_movies:
        if movie_id in tmdb_indexed.index:
            title = tmdb_indexed.loc[movie_id]["title"]
            results.append((title, score))
    
    return results

def find_recommended_content(user_vector, movie_id):
    pass

def analyze_recommended_content(user_vector):
    pass
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # --- 1. Základní statistiky a percentily ---
    print("=== STATISTIKY MATCH SCORE ===")
    stats = results["match_score"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    print(stats)
    print("\n")

    # --- 2. Analýza distribuce (Kladné vs Záporné) ---
    print("=== DISTRIBUCE SKÓRE ===")
    kladne = (results["match_score"] > 0).sum()
    zaporne = (results["match_score"] < 0).sum()
    nuly = (results["match_score"] == 0).sum()
    
    print(f"Filmy s kladným skóre (kandidáti na doporučení): {kladne}")
    print(f"Filmy se záporným skóre (aktivně penalizované):  {zaporne}")
    print(f"Filmy s nulovým skóre (žádná shoda v textu):    {nuly}")

In [ ]:
marvel_fan = build_user_profile_collab(marvel_fan_rd)
recommend_content(marvel_fan)

In [ ]:
marvel_fan_dc_hater = build_user_profile_collab(marvel_fan_dc_hater_rd)
recommend_content(marvel_fan_dc_hater)

In [ ]:
ratings_glob_mean